## Exhibit Watermark Tool Instructions

Name each PDF using the following format:

EXHIBIT NUMBER - DESCRIPTION.pdf

Examples:

EXHIBIT 11.1 - Dependence on Favorable Electric Power Agreements.pdf  
EXHIBIT 11.2 - Operations Located in Low Cost Energy Regions.pdf  
EXHIBIT 11.3 - Large Scale Electricity Use Dependent on Low Cost Power.pdf  

Rules:

• Use a hyphen to separate exhibit number and description  
• Everything BEFORE the hyphen becomes the watermark text  
• Everything AFTER the hyphen becomes the description  
• Multiple PDFs can be uploaded at once  

Accepted formats:

EXHIBIT 11.1 - Title.pdf  
EXHIBIT 11.1-Title.pdf  
11.1 - Title.pdf  

After uploading, the notebook will:

1. Stamp every page of each PDF  
2. Apply the exhibit watermark automatically  
3. Download a ZIP file containing all stamped exhibits

## Exhibit Watermark Tool Instructions

Name your files like this:

EXHIBIT 11.1 - Dependence on Favorable Electric Power Agreements.pdf

The tool reads the filename and automatically creates the watermark.

### Example

| Before | After |
|------|------|
| ![](https://imgur.com/bhJg7Oh.png) | ![](https://imgur.com/CfWwhOh.png) |

In [ ]:
#@title 🧾 Exhibit Watermark Tool (Turnkey) { display-mode: "form" }

# =========================================================
# INSTRUCTIONS
# =========================================================



!pip -q install pypdf reportlab

from pypdf import PdfReader, PdfWriter
from reportlab.pdfgen import canvas
from reportlab.lib.colors import Color
from google.colab import files
import io
import os
import textwrap
import zipfile
import re


# =========================================================
# SETTINGS
# =========================================================

show_exhibit_title = True #@param {type:"boolean"}

title_font = "Times-Roman" #@param ["Helvetica","Helvetica-Bold","Times-Roman","Times-Bold","Courier","Courier-Bold"]

title_font_size = 16 #@param {type:"number"}

title_y_offset = 28 #@param {type:"number"}

rotation = 45 #@param {type:"number"}

opacity = 0.16 #@param {type:"number"}

main_font_size = 80 #@param {type:"number"}

main_min_font_size = 40 #@param {type:"number"}

desc_wrap_width = 35 #@param {type:"number"}

desc_scale_ratio = 0.32 #@param {type:"number"}

desc_min_font_size = 14 #@param {type:"number"}

exhibit_font = "Times-Bold" #@param ["Helvetica","Helvetica-Bold","Times-Roman","Times-Bold","Courier","Courier-Bold"]

description_font = "Times-Roman" #@param ["Helvetica","Times-Roman","Courier"]

description_placement = "under_exhibit" #@param ["under_exhibit","top_header","none"]

footer_left = "" #@param {type:"string"}

footer_right = "Your names document could go here." #@param {type:"string"}

footer_font = "Times-Roman" #@param ["Helvetica","Times-Roman","Courier"]

footer_font_size = 9 #@param {type:"number"}

upload_new_pdfs = True #@param {type:"boolean"}

existing_pdf_prefix = "EXHIBIT" #@param {type:"string"}

output_folder_name = "stamped_exhibits" #@param {type:"string"}


# =========================================================
# PREP
# =========================================================

out_dir = f"/content/{output_folder_name}"
os.makedirs(out_dir, exist_ok=True)


# =========================================================
# HELPERS
# =========================================================

def clean_filename(text):
    return re.sub(r'[<>:"/\\|?*]', '', text)


def parse_filename(filename):

    base = os.path.splitext(os.path.basename(filename))[0].strip()

    base = re.sub(r"\s*-\s*", " - ", base)

    if " - " in base:
        exhibit_part, description = base.split(" - ", 1)
    else:
        exhibit_part = base
        description = ""

    exhibit_part = exhibit_part.strip()
    description = description.strip()

    if not exhibit_part.upper().startswith("EXHIBIT"):
        exhibit_part = f"EXHIBIT {exhibit_part}"

    return exhibit_part, description


def fit_main_text(c, text, font_name, start_size, min_size, max_width):

    size = start_size

    while size >= min_size:
        if c.stringWidth(text, font_name, size) <= max_width:
            return size
        size -= 2

    return min_size


def create_watermark(width, height, exhibit_text, description):

    packet = io.BytesIO()
    c = canvas.Canvas(packet, pagesize=(width, height))

    cx = width / 2
    cy = height / 2

    if show_exhibit_title:

        header_text = exhibit_text
        if description:
            header_text = f"{exhibit_text} - {description}"

        c.setFillColor(Color(0.1,0.1,0.1,alpha=0.95))
        c.setFont(title_font, title_font_size)

        title_y = height - title_y_offset
        c.drawCentredString(width/2, title_y, header_text)

    c.saveState()

    c.setFillColor(Color(0.5,0.5,0.5,alpha=opacity))

    c.translate(cx, cy)
    c.rotate(rotation)

    main_size = fit_main_text(
        c,
        exhibit_text,
        exhibit_font,
        main_font_size,
        main_min_font_size,
        width * 0.65
    )

    c.setFont(exhibit_font, main_size)
    c.drawCentredString(0,0,exhibit_text)

    if description and description_placement == "under_exhibit":

        desc_size = max(int(main_size * desc_scale_ratio), int(desc_min_font_size))
        c.setFont(description_font, desc_size)

        lines = textwrap.wrap(description, width=int(desc_wrap_width))

        start_y = -(main_size * 0.9)

        for i,line in enumerate(lines[:3]):
            c.drawCentredString(0, start_y - (i * desc_size * 1.2), line)

    c.restoreState()

    if footer_left.strip() or footer_right.strip():

        c.setFillColor(Color(0.15,0.15,0.15,alpha=0.85))
        c.setFont(footer_font, footer_font_size)

        margin = 24

        if footer_left.strip():
            c.drawString(margin, 14, footer_left.strip())

        if footer_right.strip():
            c.drawRightString(width-margin, 14, footer_right.strip())

    c.showPage()
    c.save()

    packet.seek(0)

    return PdfReader(packet).pages[0]


# =========================================================
# LOAD PDFS
# =========================================================

pdf_files = []

if upload_new_pdfs:

    print("Upload exhibit PDFs")
    uploaded = files.upload()

    pdf_files = [f"/content/{name}" for name in uploaded.keys() if name.lower().endswith(".pdf")]

else:

    for name in os.listdir("/content"):

        if name.lower().endswith(".pdf") and not name.endswith("__STAMPED.pdf"):

            if existing_pdf_prefix and not name.startswith(existing_pdf_prefix):
                continue

            pdf_files.append(f"/content/{name}")

if not pdf_files:
    raise ValueError("No PDFs found.")


# =========================================================
# STAMP PDFs
# =========================================================

outputs = []

for pdf_path in pdf_files:

    pdf_name = os.path.basename(pdf_path)

    exhibit_text, description = parse_filename(pdf_name)

    reader = PdfReader(pdf_path)
    writer = PdfWriter()

    for page in reader.pages:

        width = float(page.mediabox.width)
        height = float(page.mediabox.height)

        watermark = create_watermark(width, height, exhibit_text, description)

        page.merge_page(watermark)
        writer.add_page(page)

    safe_name = clean_filename(os.path.splitext(pdf_name)[0])

    out_name = f"{safe_name}__STAMPED.pdf"
    out_path = os.path.join(out_dir, out_name)

    with open(out_path,"wb") as f:
        writer.write(f)

    outputs.append(out_path)

    print("Stamped:", pdf_name)

# =========================================================
# OUTPUT LOCATION
# =========================================================

print(f"✓ {len(outputs)} exhibit(s) stamped")

if len(outputs) == 1:
    print("✓ Stamped PDF saved.")
else:
    print("✓ ZIP package created.")

print("")
print("Download your file from the left panel:")
print("📁 Click the folder icon on the left side of the notebook.")
print(f"📁 Open the folder: {output_folder_name}")
print("⬇ Right-click the file and choose Download.")

Upload exhibit PDFs


Saving EXHIBIT 11.1 - Dependence on Favorable Electric Power Agreements.pdf to EXHIBIT 11.1 - Dependence on Favorable Electric Power Agreements.pdf
Stamped: EXHIBIT 11.1 - Dependence on Favorable Electric Power Agreements.pdf
✓ 1 exhibit(s) stamped
✓ Stamped PDF saved.

Download your file from the left panel:
📁 Click the folder icon on the left side of the notebook.
📁 Open the folder: stamped_exhibits
⬇ Right-click the file and choose Download.
